In [3]:
!pip install OpenAI

In [4]:
from openai import OpenAI
API_KEY = "XXXXXX"
client = OpenAI(api_key=API_KEY)

In [5]:
PROMPT_TEMPLATE = """

You are an expert **Restaurant Review Emotion Classifier**.

## INPUT FORMAT
You have the following input format:
{"input": "review", "output": [{"aspect": "aspect", "polarity": "polarity", "emotion": "null"}]}

## TASK
For each aspect and its polarity provided in the input, predict the **single most dominant emotion** expressed about that specific aspect.

**CRITICAL**: Aspect and Polarity are PROVIDED in the input and MUST be copied EXACTLY as given. Do NOT change, infer, or correct polarity and aspect.

## DOMAIN: RESTAURANT REVIEWS
Focus only on restaurant dining experiences.

## ASPECT DEFINITIONS
Each review may refer to one or more of the following aspects:
- **food**: Taste, freshness, appearance, or quality of dishes, portion (lower priority). Includes: food, drinks, breakfast, lunch, and dinner.
- **staff**: Behavior, friendliness, professionalism, or attitude of employees, a person is explicitly mentioned. Includes: waiter, manager, busboy, host, waitress.
- **service**: Wait times, speed, order accuracy, payment, and reservations (not staff behavior), table and seat management.
- **price**: Perceived cost, fairness, or value for money.
- **ambience**: Sensory atmosphere – lighting, decor, noise, temperature, smell, cleanliness.
- **place**: Includes physical features both outside (geographical location, accessibility, or parking) and inside (tables, space, bar, and other parts inside the restaurant)
- **menu**: Variety of food and drinks, dietary options, readability, and item availability.
- **miscellaneous**: Overall experience or the category that does not fit any other aspect. (e.g "Reserve a cozy window seat for more privacy, or hop onto a stool at the bar to dine solo")

## EMOTION DECISION PROCESS

### STEP 1: Select emotion from ALLOWED SET for the provided aspect & polarity

ASPECT-SPECIFIC EMOTION ALLOWANCE (CRITICAL - USE THIS TABLE)

### FOR POSITIVE POLARITY:
- **ambience** → [admiration, satisfaction]
- **food** → [admiration, satisfaction]
- **menu** → [admiration, satisfaction]
- **place** → [admiration, satisfaction]
- **price** → [admiration, satisfaction]
- **service** → [admiration, satisfaction, gratitude]
- **staff** → [gratitude, admiration, satisfaction]
- **miscellaneous** → [admiration, satisfaction]

### FOR NEGATIVE POLARITY:
- **ambience** → [annoyance, disappointment]
- **food** → [disgust, disappointment, annoyance]
- **menu** → [disappointment, annoyance]
- **place** → [disappointment, annoyance, disgust]
- **price** → [disappointment, annoyance]
- **service** → [annoyance, disappointment]
- **staff** → [disappointment, annoyance]
- **miscellaneous** → [annoyance, disappointment]

### FOR NEUTRAL POLARITY (ALL ASPECTS):
- [neutral, mixed_emotions, mentioned_only]

## EMOTION DEFINITIONS & WHEN TO USE EACH

### Positive Emotions:
- **admiration** → Strong praise, impressive performance, exceeds expectations. Use for exceptional quality, skill, or thoughtfulness shown by staff/service (e.g. "best ever", "outstanding", "perfect", "amazing", "excellent", "wonderful", "professional", "remembered us", "suggested great options")
- **satisfaction** → Content approval without excitement, meets expectations, feeling relaxed and comfortable. Neutral positive experience (e.g. "pleasant", "nice", "enjoyable", "good", "happy", "comfortable", "not rushed", "not bothered")
- **gratitude** → ONLY for aspects "staff" and "service": Explicit thankfulness with words like "thank you", "grateful", "appreciate". Use ONLY when reviewer directly expresses thanks. Do NOT use for impressive service/staff - that's admiration.

### Negative Emotions:
- **disgust** → ONLY for the aspects "food" and "place": Sensory aversion; strong 'gross' reaction (e.g. "rotten", "unsafe", "greasy", "nasty", "dirty", "filthy area")
- **disappointment** → Deep and strong negative emotional response. It typically arises when expectations are not met and often concerns multiple qualities or the overall performance of an aspect (unlike annoyance) (e.g. "extremely bad", "the worst", "expected better", "disappointing","let down")
- **annoyance** → Reflects a milder, more situational negative reaction. It usually refers to one specific issue, incident, or short-lived situation within an aspect. The attitude is irritated or frustrated, but not strongly negative and not focused on the aspect. (e.g "annoying")

### Neutral Emotions:
- **neutral** → Mild emotion without explicit positive or negative reaction. Balanced or average assessment (e.g "average", "normal", "ok", "nothing special", "decent")
- **mentioned_only** → Aspect referenced factually without any opinion, emotion, or evaluation (e.g. "I ate the pasta", "get take-out next door", "they have wine")
- **mixed_emotions** → CLEAR opposing feelings about SAME aspect in same sentence/context (e.g "the pizza was very good, but the pasta was too salty")

## IMPORTANT RESTRICTIONS

1. Use "disgust" for aspects "food" and "place" only
2. Use "gratitude" for aspects "staff" and "service" only
3. Information about wrong or lacking food order should be under "service" aspect but not "food" (e.g. "Ten minutes later: no burgers and no sign of our waiter.").

## DECISION WORKFLOW

For EACH aspect in input:
1. Look at PROVIDED polarity (positive/negative/neutral)
2. Find aspect in table above to see ALLOWED emotions for that polarity
3. Choose ONLY ONE emotion from the allowed list

## OUTPUT FORMAT
Return EXACTLY this structure:
{
  "input": "<original review text>",
  "output": [
    {"aspect": "...", "polarity": "...", "emotion": "..."},
    {"aspect": "...", "polarity": "...", "emotion": "..."}
  ]
}

- Keep "input" with the full review text
- Keep "output" as an array of objects
- Each object must have: aspect, polarity (unchanged from input), and emotion (your prediction)
- Output ONLY valid JSON. No explanations, no markdown, no extra text.
"""

In [10]:
def build_prompt(input_data):
    """Build the prompt with the full input data"""
    prompt = PROMPT_TEMPLATE + f"\n\nInput:\n{json.dumps(input_data, indent=2)}\n\nOutput:\n"
    return prompt


def classify_review_emotions(input_data):
    """Call OpenAI API to classify emotions"""
    prompt = build_prompt(input_data)

    response = client.chat.completions.create(
        model="gpt-4.1-2025-04-14",
        messages=[
            {"role": "system", "content": "You are an expert Restaurant Review Emotion Classifier."},
            {"role": "user", "content": prompt}
        ],
        temperature=0  # for consistency
    )

    return response.choices[0].message.content.strip()


def parse_json_from_output(output_text):
    """Parse JSON from model output, handling markdown code blocks"""
    # Try to find JSON in markdown code blocks
    json_match = re.search(r"```json(.*?)```", output_text, re.DOTALL)
    if not json_match:
        json_match = re.search(r"```(.*?)```", output_text, re.DOTALL)

    if json_match:
        json_str = json_match.group(1).strip()
    else:
        json_str = output_text.strip()

    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        print(f"JSON decode error: {e}")
        print(f"Raw output: {output_text}")
        return None


def main():
    """Main function to process all reviews"""
    input_file = "/content/sample_data/edited_300_sample_cleaned_14jan.jsonl"
    output_file = "/content/sample_data/gpt_output.jsonl"

    with open(input_file, "r") as f_in, open(output_file, "w") as f_out:
        for i, line in enumerate(f_in):
            data = json.loads(line)

            try:
                # Call API with full input data (includes aspect + polarity)
                raw_output = classify_review_emotions(data)
                parsed_output = parse_json_from_output(raw_output)

                if parsed_output is None:
                    print(f"Skipping review #{i+1} due to parse error.")
                    continue

                # Write the result
                f_out.write(json.dumps(parsed_output) + "\n")
                f_out.flush()

                print(f"Processed review #{i+1}")

            except Exception as e:
                print(f"Error on review #{i+1}: {e}")

            # Respect rate limits
            time.sleep(1.5)

    print(f"\nProcessing complete! Output saved to: {output_file}")


if __name__ == "__main__":
    main()

Processed review #1
Processed review #2
Processed review #3
Processed review #4
Processed review #5
Processed review #6
Processed review #7
Processed review #8
Processed review #9
Processed review #10
Processed review #11
Processed review #12
Processed review #13
Processed review #14
Processed review #15
Processed review #16
Processed review #17
Processed review #18
Processed review #19
Processed review #20
Processed review #21
Processed review #22
Processed review #23
Processed review #24
Processed review #25
Processed review #26
Processed review #27
Processed review #28
Processed review #29
Processed review #30
Processed review #31
Processed review #32
Processed review #33
Processed review #34
Processed review #35
Processed review #36
Processed review #37
Processed review #38
Processed review #39
Processed review #40
Processed review #41
Processed review #42
Processed review #43
Processed review #44
Processed review #45
Processed review #46
Processed review #47
Processed review #48
P